# 第一周结束练习

为了证明您对 OpenAI API 和 Ollama 的熟悉程度，请构建一个可以回答技术问题的工具，  
并作出回应并作出解释。这是您在课程期间可以自己使用的工具！

In [ ]:
# 进口
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display


load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-or-v1"):
    print("An API key was found, but it doesn't start sk-or-v1; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


In [ ]:
# 常量
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2:1b'

open_router = "https://openrouter.ai/api/v1"
local_llm = "http://localhost:11434/v1"


In [ ]:
# 设置环境
system_prompt="""
You are a clear and patient teacher.
Your job is to explain complex topics in a way a smart 12-year-old can understand.
Rules:
- Use simple, everyday language.
- Avoid jargon. If you must use a technical word, define it clearly.
- Use short paragraphs.
- Use at least one real-world analogy.
- Give 3 key takeaways at the end in bullet points.
- Do not assume prior knowledge.
- Be accurate but simple.
- Keep the explanation under 400 words.

Structure your response like this:

1) Simple Definition  
2) How It Works (Step-by-step if needed)  
3) Real-Life Example or Analogy  
4) 3 Key Takeaways
"""

def get_user_prompt(question):
    user_prompt = f"""
    Explain the following in a way a smart 12-year-old would understand: {question}”
    """
    return user_prompt

In [ ]:
# 这是问题；输入此内容以询问新问题
question = """
What is an embedding in LLM Engineering?
"""

In [ ]:
# 让 gpt-4o-mini 接听，带流媒体
client = OpenAI(base_url=open_router, api_key=api_key)
response = client.chat.completions.create(
    model=MODEL_GPT,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": get_user_prompt(question.strip())}
    ],
    stream=True
)

# 当stream=True时，响应是块的生成器——收集内容然后显示为Markdown
result_parts = []
for chunk in response:
    if chunk.choices[0].delta.content:
        result_parts.append(chunk.choices[0].delta.content)
display(Markdown("".join(result_parts)))


In [ ]:
# 让 Llama 3.2 来回答
client = OpenAI(base_url=local_llm, api_key=api_key)
response = client.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": get_user_prompt(question)}
    ],
)

result = response.choices[0].message.content
display(Markdown(result))